# RNAfold
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import subprocess
from pathlib import Path

In [ ]:
method_name = "RNAfold"
base = Path.cwd()

# Setup directory structure
tools_dir = base.parent / "tools"
os.makedirs(tools_dir, exist_ok=True)

# Define paths for ViennaRNA
vienna_dir = tools_dir / "ViennaRNA"
vienna_install_dir = vienna_dir / "install"
vienna_build_dir = vienna_dir / "build"

# Create directories
for d in [vienna_dir, vienna_install_dir, vienna_build_dir]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Download and install ViennaRNA
if not (vienna_install_dir / 'bin' / 'RNAfold').exists():
    print("Installing ViennaRNA...")
    os.chdir(vienna_build_dir)
    
    !wget https://www.tbi.univie.ac.at/RNA/download/sourcecode/2_5_x/ViennaRNA-2.5.0.tar.gz
    !tar xzf ViennaRNA-2.5.0.tar.gz
    os.chdir('ViennaRNA-2.5.0')
    !./configure --prefix=$vienna_install_dir --without-perl --without-python
    !make -j$(nproc)
    !make install
    os.chdir('..')
    !rm -rf ViennaRNA-2.5.0 ViennaRNA-2.5.0.tar.gz
    print("ViennaRNA installed successfully")
    print(f"ViennaRNA install path: {vienna_install_dir}")
else:
    print(f"ViennaRNA already installed at {vienna_install_dir}")

In [ ]:
# Setup RNAfold executable path
rnafold_exe = vienna_install_dir / 'bin' / 'RNAfold'
# Set environment variables
os.environ['LD_LIBRARY_PATH'] = str(vienna_install_dir / 'lib') + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PATH'] = str(vienna_install_dir / 'bin') + ':' + os.environ.get('PATH', '')

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(fasta_name):
    # Prepare environment with library path
    env = os.environ.copy()
    env['LD_LIBRARY_PATH'] = str(vienna_install_dir / 'lib') + ':' + env.get('LD_LIBRARY_PATH', '')
    
    # Compute structure using subprocess
    with open('RNAfold_output.fasta', 'w') as outfile:
        result = subprocess.run([str(rnafold_exe), '-T', '37', '--noPS', fasta_name], 
                              stdout=outfile, stderr=subprocess.PIPE, 
                              text=True, timeout=1000000, env=env)
    
    if result.returncode == 0 and os.path.exists('RNAfold_output.fasta') and os.path.getsize('RNAfold_output.fasta') > 0:
        return 'RNAfold_output.fasta'
    else:
        return None

In [ ]:
out_dir = Path.cwd().parent / 'prediction'
os.makedirs(out_dir, exist_ok=True)
out_fasta_path = out_dir / (method_name + ".fasta")
if os.path.exists(out_fasta_path): os.remove(out_fasta_path)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}\t{'status'}")
successful_runs = 0

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    # Write a one-sequence fasta
    with open("RNAfold_input.fasta", "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding("RNAfold_input.fasta")
    elapsed_time = time.time() - start_time

    # Check if folding was successful
    if dot_file_name and os.path.exists(dot_file_name):
        # Concatenate outputs
        with open(dot_file_name, 'r') as infile:
            with open(out_fasta_path, 'a') as outfile:
                outfile.write(infile.read())
        successful_runs += 1
        status = "SUCCESS"
    else:
        status = "FAILED"

    print(f"{elapsed_time: .1f} s\t{status}")

    # Clean up temporary files
    for temp_file in ["RNAfold_input.fasta", "RNAfold_output.fasta"]:
        if os.path.exists(temp_file):
            os.remove(temp_file)

print(f"\nCompleted: {successful_runs}/{len(virus_ids)} sequences processed successfully")
print(f"Output file: {out_fasta_path}")